# TensorFlow Specialization - Course 3, Week 1
## Practice Notebook: Sentiment in Text


### Building a Vocabulary

The code below takes a list of sentences, then takes each word in those sentences and assigns it to an integer. This is done using the [TextVectorization()](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization) preprocessing layer and its [adapt()](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization#adapt) method.

As mentioned in the docs above, this layer does several things including:

1. Standardizing each example. The default behavior is to lowercase and strip punctuation. See its `standardize` argument for other options.
2. Splitting each example into substrings. By default, it will split into words. See its `split` argument for other options.
3. Recombining substrings into tokens. See its `ngrams` argument for reference.
4. Indexing tokens.
5. Transforming each example using this index, either into a vector of ints or a dense float vector.

Run the cells below to see this in action.

In [1]:
%pip install tensorflow

  Using cached tensorflow-2.20.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.5 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-6.32.1-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached termcolor-3.1.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached wrapt-1.17.3-cp310-cp310-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (6.4 kB)
  Using cached grpcio-1.75.1-cp310-cp310-manylinu

In [2]:
import tensorflow as tf

2025-10-13 13:35:23.170624: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-13 13:35:23.236127: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-13 13:35:25.079960: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# sample inputs
sentences = [
    'I love my dog',
    'I love my cat'
]

# initialize the layer
vectorize_layer = tf.keras.layers.TextVectorization()

# build the vocabulary
vectorize_layer.adapt(sentences)

# get the vocabulary list, ignore special tokens for now
vocabulary = vectorize_layer.get_vocabulary(include_special_tokens=False)

The resulting `vocabulary` will be a list where more frequently used words will have a lower index. By default, it will also reserve indices for special tokens but , for clarity, let's reserve that for later.

In [5]:
# print the token index
for index,word in enumerate(vocabulary):
    print(index, word)

0 my
1 love
2 i
3 dog
4 cat


If you add another sentence, you'll notice new words in the vocabulary and new punctuation is still ignored as expected.

In [6]:
# Add another input
sentences = [
    'i love my dog',
    'I, love my cat',
    'You love my dog!'
]


# Initialize the layer
vectorize_layer = tf.keras.layers.TextVectorization()

# Build the vocabulary
vectorize_layer.adapt(sentences)

# Get the vocabulary list. Ignore special tokens for now.
vocabulary = vectorize_layer.get_vocabulary(include_special_tokens=False)

In [7]:
# Print the token index
for index, word in enumerate(vocabulary):
  print(index, word)

0 my
1 love
2 i
3 dog
4 you
5 cat


Now that you see how it behaves, let's include the two special tokens. The first one at `0` is used for padding and `1` is used for out-of-vocabulary words. These are important when you use the layer to convert input texts to integer sequences. You'll see that in the next lab.

In [8]:
# get the vocabulary list
vocabulary = vectorize_layer.get_vocabulary()

# print the token index
for index, word in enumerate(vocabulary):
    print(index , word)

0 
1 [UNK]
2 my
3 love
4 i
5 dog
6 you
7 cat


That concludes this short exercise on building a vocabulary!

---

### Generating Sequences and Padding

In this lab, you will look at converting input sentences into numeric sequences. Similar to images in the previous course, you need to prepare text data with uniform size before feeding it to your model. You will see how to do these in the next sections.

#### Text to Sequences

In the previous lab, you saw how to use the `TextVectorization` layer to build a vocabulary from your corpus. It generates a list where more frequent words have lower indices.

You can then use the result to convert each of the input sentences into integer sequences. See how that's done below given a single input string.

In [12]:
# string input
sample_input = "I love my dog"

# convert the string input to an integer sequence
sequence = vectorize_layer(sample_input)

print(sequence)

tf.Tensor([6 3 2 4], shape=(4,), dtype=int64)


As shown, we simply pass in the string to the layer which already learned the vocabulary, and it will output the integer sequence as a `tf.Tensor`. In this case, the result is `[6 3 2 4]`. We can look at the token index printed above to verify that it matches the indices for each word in the input string.

For a given list of string inputs (such as the 4-item `sentences` list above), we will need to apply the layer to each input. There's more than one way to do this. Let's first use the `map()` method and see the results.

In [17]:
# convert the list to tf.data.Dataset
sentences_dataset = tf.data.Dataset.from_tensor_slices(sentences)

# define a mapping function to convert each sample input
sequences = sentences_dataset.map(vectorize_layer)
# print("sequences - ",sequences)

# print the integer sequences
for sentence, sequence in zip(sentences, sequences):
    print(f'{sentence} ---> {sequence}')

I love my dog ---> [6 3 2 4]
I love my cat ---> [ 6  3  2 10]
You love my dog! ---> [5 3 2 4]
Do you think my dog is amazing? ---> [ 9  5  7  2  4  8 11]


As you can see, each sentence is successfully transformed into an integer sequence. The problem with this is they have varying lengths so it cannot be consumed by the model right away.

#### Padding

we can get a list of varying lengths to have a uniform size by padding or truncating tokens from the sequences. Padding is more common to preserve information.

Recall that our vocabulary reserves a special token index `0` for padding. It will add that token (called post padding) if we pass in a list of string inputs to the layer. See an example below. Notice that we have the same output as above but the integer sequences are already post-padded with `0` up to the length of the longest sequence.

In [18]:
# apply the layer to the string input list
sequences_post = vectorize_layer(sentences)

# Print the results
print('INPUT:')
print(sentences)
print()

print('OUTPUT:')
print(sequences_post)

INPUT:
['I love my dog', 'I love my cat', 'You love my dog!', 'Do you think my dog is amazing?']

OUTPUT:
tf.Tensor(
[[ 6  3  2  4  0  0  0]
 [ 6  3  2 10  0  0  0]
 [ 5  3  2  4  0  0  0]
 [ 9  5  7  2  4  8 11]], shape=(4, 7), dtype=int64)


If we want pre-padding, we can use the [pad_sequences()](https://www.tensorflow.org/api_docs/python/tf/keras/utils/pad_sequences) utility to prepend a padding token to the sequences. Notice that the `padding` argument is set to `pre`. This is just for clarity. The function already has this set as the default so we can opt to drop it.

In [20]:
# pre-pad the sequences to a uniform length
# we can remove the padding argument and get the same result
sequences_pre = tf.keras.utils.pad_sequences(sequences, padding='pre')

# Print the results
print('INPUT:')
[print(sequence.numpy()) for sequence in sequences]
print()

print('OUTPUT:')
print(sequences_pre)

INPUT:
[6 3 2 4]
[ 6  3  2 10]
[5 3 2 4]
[ 9  5  7  2  4  8 11]

OUTPUT:
[[ 0  0  0  6  3  2  4]
 [ 0  0  0  6  3  2 10]
 [ 0  0  0  5  3  2  4]
 [ 9  5  7  2  4  8 11]]


2025-10-13 15:45:46.115964: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-10-13 15:45:46.148071: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


If we switch the `padding` argument to `post`, we will arrive at the same result as applying the layer directly.

The function also has a `maxlen` argument that we can use to truncate tokens from the sequences. By default, it will drop tokens in front. If we want to drop tokens at the other end, we will have to set the `truncating` argument to `post`.

In [21]:
# Post-pad the sequences and limit the size to 5.
sequences_post_trunc = tf.keras.utils.pad_sequences(sequences, maxlen=5, padding='pre')

# Print the results
print('INPUT:')
[print(sequence.numpy()) for sequence in sequences]
print()

print('OUTPUT:')
print(sequences_post_trunc)

INPUT:
[6 3 2 4]
[ 6  3  2 10]
[5 3 2 4]
[ 9  5  7  2  4  8 11]

OUTPUT:
[[ 0  6  3  2  4]
 [ 0  6  3  2 10]
 [ 0  5  3  2  4]
 [ 7  2  4  8 11]]


2025-10-13 15:46:55.113465: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Another way to prepare your sequences for prepadding is to set the TextVectorization to output a ragged tensor. This means the output will not be automatically post-padded. See the output sequences here.

In [22]:
# Set the layer to output a ragged tensor
vectorize_layer = tf.keras.layers.TextVectorization(ragged=True)

# Compute the vocabulary
vectorize_layer.adapt(sentences)

# Apply the layer to the sentences
ragged_sequences = vectorize_layer(sentences)

# Print the results
print(ragged_sequences)

<tf.RaggedTensor [[6, 3, 2, 4], [6, 3, 2, 10], [5, 3, 2, 4], [9, 5, 7, 2, 4, 8, 11]]>


With that, we can now pass it directly to the `pad_sequences()` utility.

In [23]:
# Pre-pad the sequences in the ragged tensor
sequences_pre = tf.keras.utils.pad_sequences(ragged_sequences.numpy())

# Print the results
print(sequences_pre)

[[ 0  0  0  6  3  2  4]
 [ 0  0  0  6  3  2 10]
 [ 0  0  0  5  3  2  4]
 [ 9  5  7  2  4  8 11]]


#### Out-of-vocabulary tokens

Lastly, you'll see what the other special token is for. The layer will use the token index `1` when you have input words that are not found in the vocabulary list. For example, you may decide to collect more text after your initial training and decide to not recompute the vocabulary. You will see this in action in the cell below. Notice that the token `1` is inserted for words that are not found in the list.

In [24]:
# Try with words that are not in the vocabulary
sentences_with_oov = [
    'i really love my dog',
    'my dog loves my manatee'
]

# Generate the sequences
sequences_with_oov = vectorize_layer(sentences_with_oov)

# Print the integer sequences
for sentence, sequence in zip(sentences_with_oov, sequences_with_oov):
  print(f'{sentence} ---> {sequence}')

i really love my dog ---> [6 1 3 2 4]
my dog loves my manatee ---> [2 4 1 2 1]


This concludes another introduction to text data preprocessing. So far, you've just been using dummy data. In the next exercise, you will be applying the same concepts to a real-world and much larger dataset.

---

### Tokenizing the Sarcasm Dataset

In this lab, you will apply what you've learned in the past two exercises to preprocess the [News Headlines Dataset for Sarcasm Detection](https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection). This contains news headlines which are labeled as sarcastic or not. You will revisit this dataset in later labs so it is good to be acquainted with it now.

In [26]:
%pip install tensorflow_datasets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 1.1 MB/s  0:00:04a 0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 2.1 MB/s  0:00:012.2 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 MB 24.4 MB/s  0:00:01 eta 0:00:010:01:01
  DEPRECATION: Building 'promise' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'promise'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for promise: filename=promise-2.3-py3-none-any.whl size=21484 sha256=e64c2cebeeacf7f148664d6aa20229e52abce6c45770e82581363614f0e579d9
  Stored in directory: /home/bidut/.cache/pip/wheels/54/4e/28/3ed

In [27]:
import tensorflow as tf
import json
import tensorflow_datasets as tfds
from tensorflow.keras.utils import pad_sequences

#### Download and inspect the dataset

Then, we will fetch the dataset and preview some of its elements.

In [28]:
# Download the dataset
!wget -nc https://storage.googleapis.com/tensorflow-1-public/course3/sarcasm.json

--2025-10-13 16:15:18--  https://storage.googleapis.com/tensorflow-1-public/course3/sarcasm.json
Resolving storage.googleapis.com (storage.googleapis.com)... 2404:6800:4002:804::201b, 2404:6800:4002:814::201b, 2404:6800:4002:815::201b, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|2404:6800:4002:804::201b|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5643545 (5.4M) [application/json]
Saving to: ‘sarcasm.json’

sarcasm.json        100%[===================>]   5.38M  2.45MB/s    in 2.2s    

2025-10-13 16:15:21 (2.45 MB/s) - ‘sarcasm.json’ saved [5643545/5643545]



The dataset is saved as a [JSON](https://www.json.org/json-en.html) file and we can use Python's [`json`](https://docs.python.org/3/library/json.html) module to load it into our workspace. The cell below unpacks the JSON file into a list.

In [29]:
# Load the JSON file
with open("./sarcasm.json", 'r') as f:
    datastore = json.load(f)

We can inspect a few of the elements in the list. We will notice that each element consists of a dictionary with a URL link, the actual headline, and a label named `is_sarcastic`. Printed below are two elements with contrasting labels.

In [30]:
# Non-sarcastic headline
print(datastore[0])

# Sarcastic headline
print(datastore[20000])

{'article_link': 'https://www.huffingtonpost.com/entry/versace-black-code_us_5861fbefe4b0de3a08f600d5', 'headline': "former versace store clerk sues over secret 'black code' for minority shoppers", 'is_sarcastic': 0}
{'article_link': 'https://www.theonion.com/pediatricians-announce-2011-newborns-are-ugliest-babies-1819572977', 'headline': 'pediatricians announce 2011 newborns are ugliest babies in 30 years', 'is_sarcastic': 1}


With that, you can collect the headlines because those are the string inputs that you will preprocess into numeric features.


In [38]:
# Append the headline elements into the list
sentences = [item['headline'] for item in datastore]


In [40]:
# example
sentences[0]

"former versace store clerk sues over secret 'black code' for minority shoppers"

#### Preprocessing the headlines

You can convert the sentences list above into padded sequences by using the same methods you've been using in the previous labs. The cells below will build the vocabulary, then use that to generate the list of post-padded sequences for each of the 26,709 headlines.

In [41]:
# instantiate the layer
vectorize_layer = tf.keras.layers.TextVectorization()

# build the vocabulary
vectorize_layer.adapt(sentences)

# apply the layer for post padding
post_padded_sequences = vectorize_layer(sentences)

You can view the results for a particular headline by changing the value of `index` below.

In [42]:
# Print a sample headline and sequence
index = 2
print(f'sample headline: {sentences[index]}')
print(f'padded sequence: {post_padded_sequences[index]}')
print()

# Print dimensions of padded sequences
print(f'shape of padded sequences: {post_padded_sequences.shape}')

sample headline: mom starting to fear son's web series closest thing she will have to grandchild
padded sequence: [  140   825     2   813  1100  2048   571  5057   199   139    39    46
     2 13050     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0]

shape of padded sequences: (26709, 39)


For prepadding, you have to setup the `TextVectorization` layer differently. You don't want to have the automatic postpadding shown above, and instead have sequences with variable length. Then, you will pass it to the `pad_sequences()` utility function you used in the previous lab. The cells below show one way to do it:

* First, you will initialize the `TextVectorization` layer and set its `ragged` flag to `True`. This will result in a [ragged tensor](https://www.tensorflow.org/guide/ragged_tensor) which simply means a tensor with variable-length elements. The sequences will indeed have different lengths after removing the zeroes, thus you will need the ragged tensor to contain them.

* Like before, you will use the layer's `adapt()` method to generate a vocabulary.

* Then, you will apply the layer to the string sentences to generate the integer sequences. As mentioned, this will not be post-padded.

* Lastly, you will pass this ragged tensor to the [pad_sequences()](https://www.tensorflow.org/api_docs/python/tf/keras/utils/pad_sequences) function to generate pre-padded sequences.

In [43]:
# Instantiate the layer and set the `ragged` flag to `True`
vectorize_layer = tf.keras.layers.TextVectorization(ragged=True)

# Build the vocabulary
vectorize_layer.adapt(sentences)

In [44]:
# Apply the layer to generate a ragged tensor
ragged_sequences = vectorize_layer(sentences)

In [45]:
# Print a sample headline and sequence
index = 2
print(f'sample headline: {sentences[index]}')
print(f'padded sequence: {ragged_sequences[index]}')
print()

# Print dimensions of padded sequences
print(f'shape of padded sequences: {ragged_sequences.shape}')

sample headline: mom starting to fear son's web series closest thing she will have to grandchild
padded sequence: [  140   825     2   813  1100  2048   571  5057   199   139    39    46
     2 13050]

shape of padded sequences: (26709, None)


In [46]:
# Apply pre-padding to the ragged tensor
pre_padded_sequences = pad_sequences(ragged_sequences.numpy())

# Preview the result for the 2nd sequence
pre_padded_sequences[2]

array([    0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,   140,   825,
           2,   813,  1100,  2048,   571,  5057,   199,   139,    39,
          46,     2, 13050], dtype=int32)

You can see the results for post-padded and pre-padded sequences by changing the value of `index` below.

In [47]:
# Print a sample headline and sequence
index = 2
print(f'sample headline: {sentences[index]}')
print()
print(f'post-padded sequence: {post_padded_sequences[index]}')
print()
print(f'pre-padded sequence: {pre_padded_sequences[index]}')
print()

# Print dimensions of padded sequences
print(f'shape of post-padded sequences: {post_padded_sequences.shape}')
print(f'shape of pre-padded sequences: {pre_padded_sequences.shape}')

sample headline: mom starting to fear son's web series closest thing she will have to grandchild

post-padded sequence: [  140   825     2   813  1100  2048   571  5057   199   139    39    46
     2 13050     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0]

pre-padded sequence: [    0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0   140   825     2   813  1100  2048   571  5057   199   139    39
    46     2 13050]

shape of post-padded sequences: (26709, 39)
shape of pre-padded sequences: (26709, 39)


This concludes the short demo on text data preprocessing on a relatively large dataset. Next week, you will start building models that can be trained on these output sequences. See you there!

---